# 10. Multi-Objective Ranking

This notebook implements a fixed-weight multi-objective ranking algorithm, combining the SVD Preference Score, the newly generated Composite Health Score, and Preparation Time to recommend optimal food items.

In [1]:
import pandas as pd
import numpy as np
import pickle
import os
from sklearn.preprocessing import MinMaxScaler

os.makedirs('output', exist_ok=True)

## 1. Load Models and Features

In [2]:
# Load SVD Model
with open('output/svd_model.pkl', 'rb') as f:
    svd_model = pickle.load(f)

# Load Features
features_df = pd.read_csv('food_features_engineered.csv')
if 'item_id' in features_df.columns:
    features_df.rename(columns={'item_id': 'recipeid'}, inplace=True)
features_df = features_df.drop_duplicates(subset=['recipeid']).set_index('recipeid')

print(f"Loaded {len(features_df)} feature records.")

Loaded 522517 feature records.


## 2. Normalize Objective Scores
- **Preference:** Output of SVD is mapped approximately to `[0, 1]` using `(rating - 1) / 4`.
- **Health:** `health_score` is already composite and effectively distributed between `[0, 1]`.
- **Time:** Inversely normalized `prep_time`.

In [3]:
scaler = MinMaxScaler()

features_df['health_score'].fillna(features_df['health_score'].median(), inplace=True)
features_df['prep_time'].fillna(features_df['prep_time'].median(), inplace=True)

# Time Normalization (Inverse: Faster = Higher Score)
prep_99 = features_df['prep_time'].quantile(0.99)
clipped_time = features_df['prep_time'].clip(0, prep_99)
features_df['time_score'] = 1.0 - scaler.fit_transform(clipped_time.values.reshape(-1, 1)).flatten()

item_metrics = features_df[['health_score', 'time_score', 'recipe_name', 'category']].to_dict(orient='index')
print("Item metrics ready.")

C:\Users\ishan shastri\AppData\Local\Temp\ipykernel_14512\1710878239.py:3: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  features_df['health_score'].fillna(features_df['health_score'].median(), inplace=True)
C:\Users\ishan shastri\AppData\Local\Temp\ipykernel_14512\1710878239.py:4: ChainedAssignmentError: A value is being set on a co

Item metrics ready.


## 3. Implement Fixed-Weight Multi-Objective Scoring
Here we define our function and apply a balanced weight vector (e.g. `Pref=0.5, Health=0.3, Time=0.2`).

In [4]:
def get_multi_objective_recommendations(user_id, candidate_items, w_pref=0.5, w_health=0.3, w_time=0.2, top_n=10):
    scored_candidates = []
    
    for item in candidate_items:
        # 1. Preference Score
        est_rating = svd_model.predict(user_id, item).est
        pref_score = (est_rating - 1.0) / 4.0
        
        # 2. Extract item metrics
        metrics = item_metrics.get(item, {'health_score': 0, 'time_score': 0, 'recipe_name': 'Unknown', 'category': 'Unknown'})
        
        # 3. Final Composite Score
        final_score = (w_pref * pref_score) + (w_health * metrics['health_score']) + (w_time * metrics['time_score'])
        
        scored_candidates.append({
            'recipeid': item,
            'recipe_name': metrics['recipe_name'],
            'category': metrics['category'],
            'pref_score': pref_score,
            'health_score': metrics['health_score'],
            'time_score': metrics['time_score'],
            'final_score': final_score
        })
        
    # Sort descending by final score
    scored_candidates.sort(key=lambda x: x['final_score'], reverse=True)
    
    return pd.DataFrame(scored_candidates[:top_n])

## 4. Test on a Sample User

In [5]:
# Let's pick a user and run it on a small candidate set (to save time)
sample_user = 1533  # Example user ID, replace with a known ID if needed
import random
random.seed(42)
all_items = list(item_metrics.keys())
candidate_subset = random.sample(all_items, min(1000, len(all_items)))

recs_df = get_multi_objective_recommendations(sample_user, candidate_subset)
display(recs_df)

# Save to output
recs_df.to_csv('output/sample_multi_objective_recs.csv', index=False)
print("✅ Saved sample recommendations to output/sample_multi_objective_recs.csv")

,recipeid,recipe_name,category,pref_score,health_score,time_score,final_score
0,196001,Brined Roasted Turkey,Whole Turkey,0.958080,0.877035,0.965972,0.935345
1,136534,Stuffed Chicken Breasts With Mascarpone and Wh...,Chicken Breast,0.958080,0.785265,0.965278,0.907675
2,49567,Hearty Black-Eyed Pea Salad,Vegetable,1.000000,0.697805,0.989583,0.907258
3,231466,Chicken With Shrimp,Chicken,0.958080,0.784442,0.958333,0.906039
4,364571,Gordon Ramsay's Salmon With Baked Herbs &amp; ...,European,0.958080,0.788427,0.951389,0.905846
5,125722,Hamburger Barley Vegetable Soup,Lunch/Snacks,0.996445,0.724292,0.940972,0.903705
6,110318,Satay of Beef With Peanut Sauce,Meat,0.958080,0.768261,0.968750,0.903268
7,129668,Seafood Pasta,Squid,0.958080,0.765332,0.972222,0.903084
8,469729,Chicken With Spiced Masala and Coconut Milk,Curries,0.958080,0.772876,0.958333,0.902569
9,461930,Spice Traders' Chicken Curry,Indian,0.958080,0.764288,0.968750,0.902076


✅ Saved sample recommendations to output/sample_multi_objective_recs.csv
